# Capstone EDA template

From the Unit 2 glossary's [Capstone playbook](https://github.com/CharleyHananiaGA/DSB-Unit-02/blob/main/Readme.MD#s00-capstone-playbook-the-data-stage). Change `DATA_URL`, `TARGET` and `ID_COL`, run top to bottom, and write your answers in the markdown cells.

## Step 1: setup (imports, settings, seed, paths)

_Notes and decisions:_

In [ ]:
# Step 1: setup (imports, settings, seed, paths)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
rng = np.random.default_rng(SEED)
DATA_URL = "https://raw.githubusercontent.com/CharleyHananiaGA/DSB-Unit-02/main/data/titanic-adj.csv"
TARGET, ID_COL = "Survived", "PassengerId"
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print("pandas", pd.__version__, "| numpy", np.__version__)

## Step 2: load and first look

_Notes and decisions:_

In [ ]:
# Step 2: load and first look
raw = pd.read_csv(DATA_URL)
df = raw.copy()                       # keep raw untouched
print(df.shape)
df.info()
df.sample(5, random_state=SEED)

## Step 3: data dictionary and missingness table

_Notes and decisions:_

In [ ]:
# Step 3: data dictionary and missingness table
dictionary = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unique": df.nunique(),
                           "n_missing": df.isna().sum(), "pct_missing": (df.isna().mean() * 100).round(1)})
dictionary["meaning"] = ""            # fill in by hand
assert df[ID_COL].is_unique, "ID column should identify one row each"
dictionary.sort_values("pct_missing", ascending=False)

## Step 4: clean (re-runnable functions, raw stays untouched)

_Notes and decisions:_

In [ ]:
# Step 4: clean (re-runnable functions, raw stays untouched)
def clean_data(d):
    d = d.drop_duplicates()
    d = d.assign(has_cabin=d["Cabin"].notna(),
                 family_size=d["SibSp"] + d["Parch"] + 1)
    return d.drop(columns=["Cabin", "Ticket", "Name"])

df = clean_data(raw)
print(len(raw), "->", len(df), "rows")

## Step 5: univariate EDA (every modelled column)

_Notes and decisions:_

In [ ]:
# Step 5: univariate EDA (every modelled column)
num_cols = df.select_dtypes("number").columns.drop([ID_COL, TARGET]).tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
df[num_cols].hist(bins=30, figsize=(10, 6))
plt.tight_layout(); plt.savefig(FIG_DIR / "numeric_distributions.png", dpi=120)
for col in cat_cols:
    print(df[col].value_counts(dropna=False).head(10), "\n")

## Step 6: the target and its relationships

_Notes and decisions:_

In [ ]:
# Step 6: the target and its relationships
print(df[TARGET].value_counts(normalize=True).round(3))
fig, axes = plt.subplots(1, len(cat_cols), figsize=(4 * len(cat_cols), 3))
for ax, col in zip(np.atleast_1d(axes), cat_cols):
    sns.barplot(data=df, x=col, y=TARGET, errorbar=None, ax=ax)
fig.tight_layout()
df[num_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

## Step 7: correlation, redundancy and a leakage screen

_Notes and decisions:_

In [ ]:
# Step 7: correlation, redundancy and a leakage screen
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0)
suspects = [c for c in num_cols if abs(df[c].corr(df[TARGET])) > 0.9]
print("Possible leakage (|r| > 0.9 with the target):", suspects)

## Step 8: split, preprocessing pipeline and a baseline

_Notes and decisions:_

In [ ]:
# Step 8: split, preprocessing pipeline and a baseline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = [c for c in num_cols + cat_cols]
X_train, X_test, y_train, y_test = train_test_split(df[features], df[TARGET], test_size=0.2,
                                                    random_state=SEED, stratify=df[TARGET])
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
])
baseline = cross_val_score(DummyClassifier(strategy="most_frequent"), X_train, y_train, cv=5).mean()
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
simple = cross_val_score(model, X_train, y_train, cv=5).mean()
print(f"baseline accuracy {baseline:.3f} | logistic regression {simple:.3f}")

## Step 9: save outputs and record decisions

_Notes and decisions:_

In [ ]:
# Step 9: save outputs and record decisions
Path("outputs").mkdir(exist_ok=True)
df.to_csv("outputs/clean.csv", index=False)
decisions = pd.DataFrame([
    {"step": "drop Cabin", "reason": "77% missing; kept has_cabin flag"},
    {"step": "impute Age (median, in pipeline)", "reason": "20% missing, differs by class (MAR)"},
])
decisions.to_csv("outputs/decisions.csv", index=False)
decisions